In [1]:
!hostname

cn002.delta.ncsa.illinois.edu


In [2]:
import moscot as mc

In [3]:
import scanpy as sc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import os
os.chdir('/projects/bgdb/asachan/methods/maxToki-multimodal')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
raw_dir = "/work/hdd/bgdb/asachan/datasets_hdd/SKM_multimodal_ageing/raw_files"

In [35]:
adata_rna = f'{export_dir}/rna_female_type2_counts_added.h5ad'
out_tmp = '/projects/bgdb/asachan/tmp'

## Downsampling (not random, based on dna-damage and metabolic scoring trends, so that we can focus on signals of interest)

In [10]:
rna_adata = sc.read_h5ad(adata_rna)
# filter out OM2 cells
rna_adata = rna_adata[rna_adata.obs['sample'] != 'OM2'].copy()
# filter out YM2_1 cells
rna_adata = rna_adata[rna_adata.obs['orig.ident'] != 'YM2_1'].copy()

In [11]:
#drop OM9_D* cells from rna_adata (these are Gluteus Medius cells)
rna_adata = rna_adata[~rna_adata.obs['orig.ident'].str.contains('OM9_D')]

In [12]:
rna_adata.obs['orig.ident'].value_counts()

orig.ident
OM9_N1    2324
OM9_N2    2290
OM9_N4    1526
OM9_N3    1499
YM2_2     1139
OM6_3      978
OM6_4      937
YM2_3      850
OM6_2      830
OM6_1      691
P26_2       45
P26_5       40
P26_4       26
P26_1       11
Name: count, dtype: int64

In [13]:
rna_adata.obs['sample'].value_counts()

sample
OM9    7639
OM6    3436
YM2    1989
P26     122
Name: count, dtype: int64

# adding raw counts layer for these retained cells

In [14]:
rna_adata

View of AnnData object with n_obs × n_vars = 13186 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII'
    var: 'features'
    uns: 'Annotation_colors', 'Final_annotation2_colors', 'Final_annotation3_colors', 'Final_annotation4_colors', 'Final_annotation_colors', 'age_pop_colors', 'anno_0713_colors', 'anno_0715_colors', 'fiber_class_V1_colors', 'integrated_snn_res.0.8_colors', 'integrated_snn_res.2.5_colors', 'integrated_snn_res.2_colors', 'integrated_snn_res.3_colors', 'integrated_snn_res.7_colors', 'orig.ident_colors', 'rank_genes_groups'
    obsm: 'X_umap'

In [15]:
rna_adata.X.max() # float means the counts are normalized

np.float64(4.331152716932476)

#### Strip suffix per replicate library and intersect

In [19]:
import gzip, glob, os, re
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad

raw_root = f"{raw_dir}/female_type2_maxtoki"

orig_to_lib = {
    'OM6_1':  'OM6_VL_snRNA_seq_1',
    'OM6_2':  'OM6_VL_snRNA_seq_2',
    'OM6_3':  'OM6_VL_snRNA_seq_3',
    'OM6_4':  'OM6_VL_snRNA_seq_4',
    'OM9_N1': 'OM9_VL_snRNA_seq_1',
    'OM9_N2': 'OM9_VL_snRNA_seq_2',
    'OM9_N3': 'OM9_VL_snRNA_seq_3',
    'OM9_N4': 'OM9_VL_snRNA_seq_4',
    'YM2_2':  'YM2_ST_snRNA_seq_2',
    'YM2_3':  'YM2_ST_snRNA_seq_3',
    'P26_1':  'P26_ST_snRNA_seq_1',
    'P26_2':  'P26_ST_snRNA_seq_2',
    'P26_4':  'P26_ST_snRNA_seq_3',
    'P26_5':  'P26_ST_snRNA_seq_4',
}
assert set(orig_to_lib) == set(rna_adata.obs['orig.ident'].unique())

#### convert adata cells to raw library barcodes for intersection

pat = re.compile(r'^(CELL\d+_N\d+)(_.+)$')

# 1. Find merge suffix per orig.ident
suffix_per_orig = {}
for oid in rna_adata.obs['orig.ident'].unique():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    suffixes = {pat.match(c).group(2) for c in cells}
    assert len(suffixes) == 1, f"{oid}: multiple suffixes {suffixes}"
    suffix_per_orig[oid] = suffixes.pop()

# 2. Strip the suffix → set of raw-style barcodes per orig.ident
oid_to_stripped = {}
for oid, suf in suffix_per_orig.items():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    oid_to_stripped[oid] = {c[:-len(suf)] for c in cells}

# --- helpers ---
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) >= 1, f"no file matching {pattern}"
    return hits[0]

def load_raw_barcodes(path):
    return pd.read_csv(path, header=None, sep='\t')[0].tolist()

# --- per orig.ident loader ---
def load_library(oid, lib_prefix, merge_suffix):
    bc_path   = find_one(f"{raw_root}/**/{lib_prefix}_barcodes.tsv*")
    feat_path = find_one(f"{raw_root}/**/{lib_prefix}_features.tsv*")
    mat_path  = find_one(f"{raw_root}/**/{lib_prefix}_matrix.mtx*")

    raw_bcs = load_raw_barcodes(bc_path)

    # subset assertion
    expected = oid_to_stripped[oid]
    missing  = expected - set(raw_bcs)
    assert not missing, (
        f"{oid}: {len(missing)}/{len(expected)} adata cells absent from "
        f"{os.path.basename(bc_path)}"
    )

    feats    = pd.read_csv(feat_path, header=None, sep='\t')
    gene_col = 1 if feats.shape[1] >= 2 else 0
    genes    = feats[gene_col].astype(str).tolist()

    X = sio.mmread(mat_path).T.tocsr()                   # cells × genes
    assert X.shape == (len(raw_bcs), len(genes))

    # rename raw barcodes back to adata's merge-suffixed form
    new_names = [bc + merge_suffix for bc in raw_bcs]

    a = ad.AnnData(
        X=X,
        obs=pd.DataFrame({'orig.ident': oid}, index=pd.Index(new_names)),
        var=pd.DataFrame(index=pd.Index(genes)),
    )
    a.var_names_make_unique()

    # subset adata cells from this library
    keep = rna_adata.obs_names.intersection(a.obs_names)
    assert len(keep) == (rna_adata.obs['orig.ident'] == oid).sum(), \
        f"{oid}: expected {(rna_adata.obs['orig.ident']==oid).sum()} cells, got {len(keep)}"
    a = a[keep].copy()
    return a

In [20]:
# Quick check of barcodes in adata being a complete subset of barcodes in raw library files (P26, YM2, OM6, OM9)
sample_name = 'OM9'
for oid in [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']:
    print(oid, len(oid_to_stripped[oid]),
          'cells:', list(oid_to_stripped[oid])[:3])

sample_raw_dir = f"{raw_root}/{sample_name}"
sample_libs = [f'{sample_name}_VL_snRNA_seq_1', f'{sample_name}_VL_snRNA_seq_2',
            f'{sample_name}_VL_snRNA_seq_3', f'{sample_name}_VL_snRNA_seq_4']
sample_oids = [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']

# Load raw barcode sets
raw_sample = {}
for lib in sample_libs:
    raw_sample[lib] = set(load_raw_barcodes(f"{sample_raw_dir}/{lib}_barcodes.tsv.gz"))
    print(f"{lib}: {len(raw_sample[lib])} barcodes")

# 4 x 4 subset table: rows = orig.ident, cols = raw library
rows = []
for oid in sample_oids:
    expected = oid_to_stripped[oid]   # adata cells (suffix-stripped) for this orig.ident
    row = {'orig.ident': oid, 'n_adata': len(expected)}
    for lib in sample_libs:
        n = len(expected & raw_sample[lib])
        row[lib] = f"{n}/{len(expected)}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

OM9_N1 2324 cells: ['CELL10384_N1', 'CELL2408_N1', 'CELL5345_N1']
OM9_N2 2290 cells: ['CELL1089_N1', 'CELL2994_N1', 'CELL2260_N1']
OM9_N3 1499 cells: ['CELL2994_N1', 'CELL4500_N1', 'CELL4290_N1']
OM9_N4 1526 cells: ['CELL3749_N1', 'CELL5038_N1', 'CELL2408_N1']
OM9_VL_snRNA_seq_1: 10924 barcodes
OM9_VL_snRNA_seq_2: 10531 barcodes
OM9_VL_snRNA_seq_3: 9131 barcodes
OM9_VL_snRNA_seq_4: 7788 barcodes
orig.ident  n_adata OM9_VL_snRNA_seq_1 OM9_VL_snRNA_seq_2 OM9_VL_snRNA_seq_3 OM9_VL_snRNA_seq_4
    OM9_N1     2324          2324/2324          2143/2324          1774/2324          1693/2324
    OM9_N2     2290          2231/2290          2290/2290          1707/2290          1640/2290
    OM9_N3     1499          1174/1499          1076/1499          1499/1499          1275/1499
    OM9_N4     1526          1297/1526          1196/1526          1308/1526          1526/1526


In [21]:
# --- load all libraries ---
per_lib = {}
for oid, lib in orig_to_lib.items():
    a = load_library(oid, lib, suffix_per_orig[oid])
    per_lib[oid] = a
    print(f"{oid:7s} ← {lib:25s}  {a.shape}")

OM6_1   ← OM6_VL_snRNA_seq_1         (691, 38285)
OM6_2   ← OM6_VL_snRNA_seq_2         (830, 38397)
OM6_3   ← OM6_VL_snRNA_seq_3         (978, 38571)
OM6_4   ← OM6_VL_snRNA_seq_4         (937, 39372)
OM9_N1  ← OM9_VL_snRNA_seq_1         (2324, 44530)
OM9_N2  ← OM9_VL_snRNA_seq_2         (2290, 44999)
OM9_N3  ← OM9_VL_snRNA_seq_3         (1499, 43958)
OM9_N4  ← OM9_VL_snRNA_seq_4         (1526, 43648)
YM2_2   ← YM2_ST_snRNA_seq_2         (1139, 36613)
YM2_3   ← YM2_ST_snRNA_seq_3         (850, 31903)
P26_1   ← P26_ST_snRNA_seq_1         (11, 26288)
P26_2   ← P26_ST_snRNA_seq_2         (45, 34118)
P26_4   ← P26_ST_snRNA_seq_3         (26, 32396)
P26_5   ← P26_ST_snRNA_seq_4         (40, 30509)


In [22]:
# --- concat per-library AnnDatas ---
raw_all = ad.concat(
    list(per_lib.values()),
    join='outer',           # union of genes across libraries
    label='lib_src',
    index_unique=None,      # cells already globally unique via merge suffix
    merge='same',
)
print(f"concat: {raw_all.shape}")

# --- align to adata cell order ---
assert set(raw_all.obs_names) == set(rna_adata.obs_names)
raw_all = raw_all[rna_adata.obs_names].copy()

concat: (13186, 51266)


In [23]:
# --- align gene axis to adata.var_names ---
gene_loc = pd.Index(raw_all.var_names).get_indexer(rna_adata.var_names)
present  = gene_loc >= 0
n_missing = (~present).sum()
print(f"genes shared: {present.sum()}/{rna_adata.n_vars}  (missing: {n_missing})")

src      = raw_all.X.tocsc()[:, gene_loc[present]]
counts   = sp.lil_matrix((rna_adata.n_obs, rna_adata.n_vars), dtype=src.dtype)
counts[:, np.where(present)[0]] = src
counts   = counts.tocsr()

rna_adata.layers['counts'] = counts

genes shared: 47230/48355  (missing: 1125)


/tmp/ipykernel_3606783/3842050276.py:12: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  rna_adata.layers['counts'] = counts


In [24]:
# --- verifications ---
totals = np.asarray(counts.sum(axis=1)).ravel()
nfeat  = np.asarray((counts > 0).sum(axis=1)).ravel()
sub    = counts[:1000].toarray()

print(f"integer?                        {(sub == sub.astype(int)).all()}")
print(f"max value:                      {sub.max()}")
print(f"corr(layer.sum, nCount_RNA)   = {np.corrcoef(totals, rna_adata.obs['nCount_RNA'])[0,1]:.6f}")
print(f"corr(nFeat_layer, nFeature_RNA) = {np.corrcoef(nfeat,  rna_adata.obs['nFeature_RNA'])[0,1]:.6f}")
print(f"max |Δ nCount|:  {np.abs(totals - rna_adata.obs['nCount_RNA'].values).max()}")
print(f"max |Δ nFeature|:{np.abs(nfeat  - rna_adata.obs['nFeature_RNA'].values).max()}")

integer?                        False
max value:                      3344.5287706344816
corr(layer.sum, nCount_RNA)   = 0.836601
corr(nFeat_layer, nFeature_RNA) = 0.999997
max |Δ nCount|:  15818.092477249964
max |Δ nFeature|:24


## Inspecting the prepared adata rna object

In [41]:
rna_adata

AnnData object with n_obs × n_vars = 13186 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count'
    var: 'features'
    uns: 'rank_genes_groups'
    obsm: 'X_umap'
    layers: 'counts', 'lognorm', 'counts_float'

In [42]:
rna_adata.X = rna_adata.layers['counts']
display(rna_adata.X.max())

np.int32(3345)

In [39]:
import numpy as np
import scipy.sparse as sp

# Preserve original
rna_adata.layers['counts_float'] = rna_adata.layers['counts']

# Rounded integer version
counts = rna_adata.layers['counts']
if sp.issparse(counts):
    counts_int = counts.copy()
    counts_int.data = np.round(counts_int.data).astype(np.int32)
    counts_int.eliminate_zeros()
    counts_int = counts_int.tocsr()
else:
    counts_int = np.round(counts).astype(np.int32)
    counts_int = sp.csr_matrix(counts_int)

rna_adata.layers['counts'] = counts_int

# Verify
print(f"dtype: {counts_int.dtype}")
print(f"sparse: {sp.issparse(counts_int)}, format: {counts_int.format}")
print(f"max: {counts_int.max()}, min: {counts_int.data.min() if counts_int.nnz else 0}")
print(f"nnz: {counts_int.nnz} (was {counts.nnz})")
print(f"memory: {counts_int.data.nbytes / 1e6:.1f} MB")

dtype: int32
sparse: True, format: csr
max: 3345, min: 1
nnz: 26430923 (was 26973556)
memory: 105.7 MB


In [40]:
import pandas as pd
import numpy as np
import anndata

# 1. Disable Arrow-backed strings globally for this session
pd.set_option('future.infer_string', False)
pd.set_option('mode.string_storage', 'python')

# 2. Diagnose
print(f"anndata version:    {anndata.__version__}")
print(f"future.infer_string: {pd.get_option('future.infer_string')}")
print(f"mode.string_storage: {pd.get_option('mode.string_storage')}")

def nuke_arrow_full(df):
    df = df.copy()
    # Index
    df.index = pd.Index(pd.array(df.index.tolist(), dtype=object),
                        name=df.index.name)
    # Columns
    df.columns = pd.Index(pd.array(df.columns.tolist(), dtype=object))
    # Each column
    for col in list(df.columns):
        s = df[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            # Rebuild categorical with object-dtype categories
            cats = pd.array(s.cat.categories.tolist(), dtype=object)
            new_cats = pd.Index(cats)
            df[col] = pd.Categorical(
                s.astype(object),                # round-trip to plain strings
                categories=new_cats,
                ordered=s.cat.ordered,
            )
        elif 'arrow' in type(s.array).__name__.lower() or 'string' in str(s.dtype).lower():
            df[col] = pd.array(s.tolist(), dtype=object)
    return df

rna_adata.obs = nuke_arrow_full(rna_adata.obs)
rna_adata.var = nuke_arrow_full(rna_adata.var)

# Verify all categoricals
for col in rna_adata.obs.columns:
    s = rna_adata.obs[col]
    if isinstance(s.dtype, pd.CategoricalDtype):
        backing = type(s.cat.categories.array).__name__
        print(f"obs.{col}: categories backing = {backing}")

# Same .uns gotcha — color arrays etc. live there and can also be ArrowStringArray
# Just drop the entire `_colors` set; scanpy regenerates them on demand
for k in list(rna_adata.uns.keys()):
    if k.endswith('_colors'):
        del rna_adata.uns[k]

out_path = f"{export_dir}/rna_female_type2_counts_added.h5ad"
rna_adata.write_h5ad(out_path)
print(f"saved: {out_path}")

/tmp/ipykernel_3606783/4187084786.py:10: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"anndata version:    {anndata.__version__}")


anndata version:    0.12.10
future.infer_string: False
mode.string_storage: python
obs.orig.ident: categories backing = NumpyExtensionArray
obs.sample: categories backing = NumpyExtensionArray
obs.tech: categories backing = NumpyExtensionArray
obs.Sex: categories backing = NumpyExtensionArray
obs.Country: categories backing = NumpyExtensionArray
obs.age_pop: categories backing = NumpyExtensionArray
obs.Annotation: categories backing = NumpyExtensionArray
obs.bead_count: categories backing = NumpyExtensionArray
saved: /work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_female_type2_counts_added.h5ad
